# Azure Content Understanding Service Demo

This notebook demonstrates how to use the Azure Content Understanding service (part of Azure Foundry) to perform OCR on PDF documents and images.

## Setup

Before running this notebook, ensure you have:
1. Installed required packages: `pip install -r requirements.txt`
2. Set up your Azure credentials in a `.env` file:
   ```
   AZURE_FOUNDRY_ENDPOINT=https://your-endpoint.azure.com
   AZURE_FOUNDRY_API_KEY=your-api-key
   ```
3. Generated sample documents by running: `python generate_samples.py`

## Import Required Libraries

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from pathlib import Path
import time

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")

## Configure Azure Foundry Credentials

In [ ]:
# Azure Foundry Content Understanding Service Configuration
AZURE_ENDPOINT = os.getenv('AZURE_FOUNDRY_ENDPOINT', 'https://your-endpoint.azure.com')
API_KEY = os.getenv('AZURE_FOUNDRY_API_KEY', 'your-api-key')

# Verify configuration
if AZURE_ENDPOINT == 'https://your-endpoint.azure.com' or API_KEY == 'your-api-key':
    print("⚠️  Warning: Please configure your Azure credentials in .env file")
else:
    print("✓ Azure credentials loaded successfully")
    print(f"Endpoint: {AZURE_ENDPOINT}")

## Main Function: Process File with Content Understanding Service

This function takes a file path as input and sends it to Azure Foundry Content Understanding service for OCR processing.

In [ ]:
def process_file_with_content_understanding(file_path):
    """
    Process a file (PDF or image) using Azure Foundry Content Understanding service.
    
    Args:
        file_path (str): Path to the file to process
        
    Returns:
        dict: JSON response from the service containing extracted content
    """
    # Validate file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    # Get file information
    file_name = os.path.basename(file_path)
    file_size = os.path.getsize(file_path)
    
    print(f"Processing file: {file_name}")
    print(f"File size: {file_size} bytes")
    
    try:
        # Prepare the API request
        headers = {
            'api-key': API_KEY,
            'Content-Type': 'application/json'
        }
        
        # Read file content
        with open(file_path, 'rb') as f:
            file_content = f.read()
        
        # Prepare multipart form data
        files = {
            'file': (file_name, file_content)
        }
        
        # Remove Content-Type header for multipart/form-data (requests will set it automatically)
        headers_for_upload = {
            'api-key': API_KEY
        }
        
        # Construct the API endpoint
        # Note: This endpoint structure is a typical pattern for Azure services
        # Adjust based on your actual Azure Foundry Content Understanding endpoint
        api_url = f"{AZURE_ENDPOINT}/contentunderstanding/documentIntelligence:analyze"
        
        print(f"Sending request to: {api_url}")
        
        # Send POST request to Azure Foundry
        response = requests.post(
            api_url,
            headers=headers_for_upload,
            files=files,
            params={'api-version': '2023-10-31-preview'},
            timeout=60
        )
        
        # Check if request was successful
        response.raise_for_status()
        
        # Parse JSON response
        result = response.json()
        
        print(f"✓ Successfully processed {file_name}")
        print(f"Response status: {response.status_code}")
        
        return result
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Error processing file: {str(e)}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response status: {e.response.status_code}")
            print(f"Response body: {e.response.text}")
        raise
    except Exception as e:
        print(f"✗ Unexpected error: {str(e)}")
        raise

print("Function defined successfully!")

## Helper Function: Display Results

In [ ]:
def display_results(result, file_name):
    """
    Display the results from Content Understanding service in a readable format.
    
    Args:
        result (dict): JSON response from the service
        file_name (str): Name of the processed file
    """
    print("="*70)
    print(f"Results for: {file_name}")
    print("="*70)
    
    # Pretty print the JSON response
    print(json.dumps(result, indent=2))
    
    print("="*70)
    
    # Extract and display key information if available
    if 'content' in result:
        print("\nExtracted Text Content:")
        print("-" * 70)
        print(result['content'])
    
    if 'pages' in result:
        print(f"\nNumber of pages: {len(result['pages'])}")
    
    if 'tables' in result:
        print(f"Number of tables detected: {len(result['tables'])}")
    
    if 'keyValuePairs' in result:
        print(f"Number of key-value pairs found: {len(result['keyValuePairs'])}")

print("Display function defined successfully!")

## Example 1: Process a PDF Invoice

In [ ]:
# Process sample invoice PDF
invoice_path = "sample_documents/sample_invoice.pdf"

try:
    result = process_file_with_content_understanding(invoice_path)
    display_results(result, "sample_invoice.pdf")
except Exception as e:
    print(f"Error: {e}")

## Example 2: Process a PDF Receipt

In [ ]:
# Process sample receipt PDF
receipt_path = "sample_documents/sample_receipt.pdf"

try:
    result = process_file_with_content_understanding(receipt_path)
    display_results(result, "sample_receipt.pdf")
except Exception as e:
    print(f"Error: {e}")

## Example 3: Process an Image (Business Card)

In [ ]:
# Process business card image
business_card_path = "sample_documents/business_card.png"

try:
    result = process_file_with_content_understanding(business_card_path)
    display_results(result, "business_card.png")
except Exception as e:
    print(f"Error: {e}")

## Example 4: Process an Image (Sign with Text)

In [ ]:
# Process sign text image
sign_path = "sample_documents/sign_text.png"

try:
    result = process_file_with_content_understanding(sign_path)
    display_results(result, "sign_text.png")
except Exception as e:
    print(f"Error: {e}")

## Batch Processing: Process All Sample Documents

In [ ]:
# Process all files in the sample_documents directory
sample_dir = "sample_documents"

if os.path.exists(sample_dir):
    files = [f for f in os.listdir(sample_dir) if f.endswith(('.pdf', '.png', '.jpg', '.jpeg'))]
    
    print(f"Found {len(files)} files to process\n")
    
    results_collection = {}
    
    for file_name in files:
        file_path = os.path.join(sample_dir, file_name)
        print(f"\n{'='*70}")
        print(f"Processing: {file_name}")
        print(f"{'='*70}")
        
        try:
            result = process_file_with_content_understanding(file_path)
            results_collection[file_name] = result
            print(f"✓ Successfully processed {file_name}\n")
            
            # Add a small delay between requests to avoid rate limiting
            time.sleep(1)
            
        except Exception as e:
            print(f"✗ Failed to process {file_name}: {e}\n")
            results_collection[file_name] = {"error": str(e)}
    
    print(f"\n{'='*70}")
    print("Batch Processing Complete")
    print(f"{'='*70}")
    print(f"Total files processed: {len(results_collection)}")
    print(f"Successful: {sum(1 for r in results_collection.values() if 'error' not in r)}")
    print(f"Failed: {sum(1 for r in results_collection.values() if 'error' in r)}")
    
else:
    print(f"Directory '{sample_dir}' not found. Please run generate_samples.py first.")

## Save Results to JSON File

In [ ]:
# Save all results to a JSON file
output_file = "content_understanding_results.json"

if 'results_collection' in locals() and results_collection:
    with open(output_file, 'w') as f:
        json.dump(results_collection, f, indent=2)
    
    print(f"✓ Results saved to {output_file}")
    print(f"File size: {os.path.getsize(output_file)} bytes")
else:
    print("No results to save. Please run the batch processing cell first.")

## Summary

This notebook demonstrates:
1. ✓ How to configure Azure Foundry Content Understanding service credentials
2. ✓ A simple Python function that takes a file path as input
3. ✓ Sending PDF and image files to the service
4. ✓ Receiving and displaying JSON responses
5. ✓ Batch processing multiple documents
6. ✓ Saving results to a JSON file

### Next Steps
- Customize the API endpoint and parameters for your specific Azure Foundry setup
- Add error handling and retry logic for production use
- Implement additional processing of the JSON responses (e.g., extract specific fields)
- Add support for more file formats
- Integrate with downstream applications or databases